# Neoantigen Peptide-MHC Structure Prediction

Predicts 3D structures of neoantigen peptide-MHC Class I complexes using **ColabFold** (AlphaFold2-Multimer).

**Data source:** HCC1395 breast cancer cell line — top neoantigens from pVACseq (NetMHCpanEL + BigMHC_EL).

**Runtime:** ~5-15 min per complex on T4 GPU. 9 unique complexes = ~1-2 hours total.

**How to run:** Runtime > Run all. Or run cells one by one.

---

In [ ]:
#@title 1. Install ColabFold (~3 min)
%%capture
!pip install "colabfold[alphafold] @ git+https://github.com/sokrypton/ColabFold"
!pip install py3Dmol
import os
os.makedirs("structures/queries", exist_ok=True)
os.makedirs("structures/predictions", exist_ok=True)
print("Setup complete")

In [ ]:
#@title 2. HCC1395 Neoantigens (from pVACseq results)

# Top neoantigens from HCC1395 breast cancer cell line
# Predicted by NetMHCpanEL + BigMHC_EL, ranked by binding percentile
# SLC25A30 appears twice (binds both HLA-C*06:02 and HLA-C*07:01) — predict once

neoantigens = [
    # (name, mutant_peptide, wildtype_peptide, hla_allele, percentile, vaf, expression)
    ("TLN2",      "TPKFKQQL",  "TPEFKQQL",  "HLA-B*08:01", 0.01, 0.452, 20.0),
    ("TRPM7",     "ELHPRITQL", "ELHPRIKQL", "HLA-B*08:01", 0.01, 0.500, 28.6),
    ("SMOX",      "VLKRKYTSF", "VLKRQYTSF", "HLA-B*08:01", 0.01, 0.600, 34.5),
    ("PRELID2",   "NMAIRSHRL", "NMAIRSHCL", "HLA-B*08:01", 0.02, 0.324, 7.1),
    ("MAP7D1",    "KEKPIPQEP", "KEEPIPQEP", "HLA-B*45:01", 0.03, 0.361, 93.2),
    ("SZT2",      "RRLHLPRHV", "RRLHLPGHV", "HLA-C*06:02", 0.03, 0.338, 34.7),
    ("TESK1",     "YSLPRAAAL", "HSLPRAAAL", "HLA-B*08:01", 0.03, 1.000, 9.2),
    ("SLC25A30",  "TRIMNQRVL", "TRMMNQRVL", "HLA-C*06:02", 0.03, 0.451, 20.3),
    ("ZNF548",    "VVFEYVAIY", "VVFEDVAIY", "HLA-A*29:02", 0.03, 0.468, 13.4),
]

print(f"Loaded {len(neoantigens)} neoantigen-MHC complexes to predict")
for name, pep, wt, hla, pct, vaf, expr in neoantigens:
    mut_pos = [i for i,(a,b) in enumerate(zip(pep,wt)) if a!=b]
    print(f"  {name:10s} {pep} (pos {mut_pos}) -> {hla} pctile={pct} VAF={vaf} expr={expr}")

In [ ]:
#@title 3. MHC Class I sequences (HLA alpha chains + B2M)

# Beta-2-microglobulin — invariant light chain of all MHC Class I
# UniProt: P61769 (mature form, signal peptide removed)
B2M = (
    "IQRTPKIQVYSRHPAENGKSNFLNCYVSGFHPSDIEVDLLKNGERIEKVEHSDLSFSKDW"
    "SFYLLYYTEFTPTEKDEYACRVNHVTLSQPKIVKWDRDM"
)

# HLA Class I alpha chain extracellular domains (alpha1 + alpha2 + alpha3)
# Source: IPD-IMGT/HLA Database, mature sequences
HLA_ALPHA = {
    "HLA-A*29:02": (
        "GSHSMRYFYTSMSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRMEPRAPWIEQEGPEY"
        "WDQETRNVKAHSQTDRANLGTLRGYYNQSEAGSHTIQIMYGCDVGSDGRFLRGYRQDAY"
        "DGKDYIALNEDLRSWTAADMAAQITKRKWEAAHEAEQLRAYLDGTCVEWLRRYLENGKE"
        "TLQRTDPPKTHMTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPA"
        "GDGTFQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRWE"
    ),
    "HLA-B*08:01": (
        "GSHSMRYFYTSVSRPGRGEPRFISVGYVDDTQFVRFDSDAASPRGEPRAPWIEQEGPEY"
        "WDRNTQIFKTNTQTYRESLRNLRGYYNQSEAGSHTWQRMYGCDVGPDGRLLRGHDQYAY"
        "DGKDYIALNEDLRSWTAADTAAQITQRKWEAARVAEQLRAYLEGLCVEWLRRYLENGKE"
        "TLQRADPPKTHVTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPA"
        "GDRTFQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRWE"
    ),
    "HLA-B*45:01": (
        "GSHSMRYFYTSVSRPGRGEPRFISVGYVDDTQFVRFDSDAASPRGEPRAPWIEQEGPEY"
        "WDRNTQIYKAQAQTDRESLRNLRGYYNQSEAGSHTWQRMYGCDVGPDGRLLRGHDQSAY"
        "DGKDYIALNEDLRSWTAADTAAQITQRKWEAARVAEQLRAYLEGLCVEWLRRYLENGKE"
        "TLQRADPPKTHVTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPA"
        "GDRTFQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRWE"
    ),
    "HLA-C*06:02": (
        "GSHSMRYFSTSVSRPGSGEPRFISVGYVDDTQFVRFDSDAASPRMEPRAPWIEQEGPEY"
        "WDRETQKYKAQAQTDRESLRNLRGYYNQSEAGSHIIQRMYGCDLGPDGRLLRGHDQSAY"
        "DGKDYIALNEDLRSWTAADTAAQITQRKLEAARVAEQLRAYLEGECVEWLRRYLENGKE"
        "TLQRADPPKTHVTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPA"
        "GDRTFQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRWE"
    ),
    "HLA-C*07:01": (
        "GSHSMRYFSTSVSRPGSGEPRFISVGYVDDTQFVRFDSDAASPRMEPRAPWIEQEGPEY"
        "WDRETQISKTNTQTYRESLRNLRGYYNQSEAGSHIIQRMYGCDVGPDGRLLRGHDQSAY"
        "DGKDYIALNEDLRSWTAADTAAQITQRKWEAARVAEQLRAYLEGECVEWLRRYLENGKE"
        "TLQRADPPKTHVTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPA"
        "GDRTFQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLRWE"
    ),
}

print(f"HLA sequences: {len(HLA_ALPHA)} alleles")
print(f"B2M: {len(B2M)} residues")
for allele, seq in HLA_ALPHA.items():
    print(f"  {allele}: {len(seq)} residues")

In [ ]:
#@title 4. Prepare AlphaFold2-Multimer input files

# Each complex = 3 chains separated by ':'
# Chain A: HLA alpha chain
# Chain B: Beta-2-microglobulin
# Chain C: Neoantigen peptide

import os

for name, peptide, wt, hla, *_ in neoantigens:
    hla_key = hla
    if hla_key not in HLA_ALPHA:
        print(f"  SKIP {name}: {hla_key} not in database")
        continue

    complex_seq = f"{HLA_ALPHA[hla_key]}:{B2M}:{peptide}"
    total_res = len(HLA_ALPHA[hla_key]) + len(B2M) + len(peptide)

    fasta_path = f"structures/queries/{name}.fasta"
    with open(fasta_path, "w") as f:
        f.write(f">{name}_{hla.replace('*','').replace(':','')}\n{complex_seq}\n")

    print(f"  {name:10s} + {hla:16s} -> {total_res} residues ({fasta_path})")

print(f"\nReady: {len(os.listdir('structures/queries'))} FASTA files")
print(f"Estimated runtime: ~{len(neoantigens) * 10} min on T4 GPU")

In [ ]:
#@title 5. Run ColabFold (~10 min per complex)

import os

queries = sorted([f for f in os.listdir("structures/queries") if f.endswith(".fasta")])
total = len(queries)

for i, fasta in enumerate(queries, 1):
    name = fasta.replace(".fasta", "")
    pred_dir = f"structures/predictions/{name}"

    # Skip if already predicted
    if os.path.exists(pred_dir) and any(f.endswith(".pdb") for f in os.listdir(pred_dir)):
        print(f"[{i}/{total}] {name} — already done, skipping")
        continue

    print(f"\n{'='*60}")
    print(f"[{i}/{total}] Predicting: {name}")
    print(f"{'='*60}")

    !colabfold_batch \
        "structures/queries/{fasta}" \
        "{pred_dir}" \
        --num-models 3 \
        --amber \
        --num-recycle 3 \
        --model-type alphafold2_multimer_v3

print(f"\n{'='*60}")
print(f"All {total} predictions complete!")
print(f"{'='*60}")

In [ ]:
#@title 6. Visualize structures (interactive 3D)

import py3Dmol
from pathlib import Path
from IPython.display import display, HTML

def show_complex(pdb_path, title):
    with open(pdb_path) as f:
        pdb = f.read()
    view = py3Dmol.view(width=800, height=500)
    view.addModel(pdb, "pdb")
    # HLA alpha = blue, B2M = cyan, peptide = red
    view.setStyle({"chain": "A"}, {"cartoon": {"color": "#4477AA", "opacity": 0.8}})
    view.setStyle({"chain": "B"}, {"cartoon": {"color": "#66CCEE", "opacity": 0.6}})
    view.setStyle({"chain": "C"}, {
        "cartoon": {"color": "#EE6677"},
        "stick":   {"color": "#EE6677", "radius": 0.15}
    })
    view.zoomTo({"chain": "C"})  # Focus on peptide in binding groove
    display(HTML(f"<h3>{title}</h3>"))
    display(HTML("<p>Blue=HLA, Cyan=B2M, <b style='color:#EE6677'>Red=Neoantigen peptide</b></p>"))
    return view.show()

for name, peptide, wt, hla, pct, vaf, expr in neoantigens:
    pred_dir = Path(f"structures/predictions/{name}_{hla.replace('*','').replace(':','')}")
    if not pred_dir.exists():
        pred_dir = Path(f"structures/predictions/{name}")

    pdbs = sorted(pred_dir.glob("*rank_001*.pdb")) if pred_dir.exists() else []
    if pdbs:
        title = f"{name}: {peptide} + {hla} (pctile={pct}, VAF={vaf}, expr={expr})"
        show_complex(str(pdbs[0]), title)
    else:
        print(f"No PDB for {name}")

In [ ]:
#@title 7. Confidence scores + download

import json
import shutil
from pathlib import Path
from google.colab import files

print(f"{'Name':<15} {'Peptide':<12} {'HLA':<16} {'pLDDT':>8} {'pTM':>8} {'ipTM':>8}")
print("-" * 75)

for name, peptide, wt, hla, *_ in neoantigens:
    pred_dir = Path(f"structures/predictions/{name}_{hla.replace('*','').replace(':','')}")
    if not pred_dir.exists():
        pred_dir = Path(f"structures/predictions/{name}")

    score_files = sorted(pred_dir.glob("*scores_rank_001*.json")) if pred_dir.exists() else []
    if score_files:
        with open(score_files[0]) as f:
            sc = json.load(f)
        plddt = sc.get("plddt", 0)
        if isinstance(plddt, list): plddt = sum(plddt) / len(plddt)
        ptm = sc.get("ptm", 0)
        iptm = sc.get("iptm", 0)
        print(f"{name:<15} {peptide:<12} {hla:<16} {plddt:>8.1f} {ptm:>8.3f} {iptm:>8.3f}")
    else:
        print(f"{name:<15} {peptide:<12} {hla:<16} {'N/A':>8} {'N/A':>8} {'N/A':>8}")

print()
print("pLDDT > 70 = confident structure")
print("ipTM > 0.6 = confident interface (peptide-MHC contact)")

# Package for download
shutil.make_archive("HCC1395_neoantigen_structures", "zip", "structures/predictions")
print(f"\nDownloading: HCC1395_neoantigen_structures.zip")
files.download("HCC1395_neoantigen_structures.zip")

---

## Interpretation

**Good candidates** have:
- ipTM > 0.6 — peptide sits stably in MHC groove
- pLDDT > 70 — overall structure is confident
- Mutation at an anchor position (pos 2 or C-terminus for Class I) may reduce binding — check visually

**Compare mutant vs wildtype:** If the mutant peptide binds MHC much better than wildtype, the immune system is more likely to recognize it as foreign.

**Next step:** Take high-confidence candidates to wet-lab validation (peptide-MHC tetramer assay, ELISPOT).